# 12.f — Ad hoc test: the change of the suitability index with the AMOC weakening, and what "robust" and "uncertain" mean

**Ad hoc test, outside the suitability chain**: it reads the delta BIO variables written by 12.a (the whole runs and
their two halves, nothing recomputed from the daily climate), the BIO of 1.1.d, the soil of 2.g and the SVM of 3.g,
and writes only in `<out_root>/test_probability_change/<route>/<set>/`.

**Questions.**
1. The binary maps (6.b, 12.b) show only the cells that switch class. Where does the AMOC weakening make the climate
   more or less favourable, also where the cells do not switch?
2. Does it move land between levels of suitability (e.g. from highly suitable to marginal) without changing the
   binary area?
3. A remark on 12.b: are the *robust losses* (red) land that is clearly lost, and the *uncertain losses* (orange) land
   whose suitability is degraded but not necessarily lost?

**Suitability index.** s = the SVM probability at equal priors (0 where the climate is novel) on the cells that pass
the fixed floors (ECOCROP TMIN of the lower part of the cell, soil ≥ 10), 0 elsewhere; s ≥ 0.5 = suitable (6.b). It
measures how typical the climate is of the cells where the crop is grown, not a yield: its classes are *climatic
similarity* classes, named after the GAEZ but not GAEZ suitability classes.

**Method.**
1. *Change of the index*: Δs = s(hosing) − s(no hosing), whole runs (49 seasons). Noise per cell from the halves of the
   four runs: SE = √(mean d² / 2), d = s(H1) − s(H2) (as 12.a, cell by cell). A **robust** change has the same sign in
   the whole run and in both halves, |Δs| > 2 SE and |Δs| ≥ `MIN_CHANGE` (0.1): where the index is close to 0 or 1 the
   noise is near zero, so without a minimum size a change of 0.98 → 0.97 would count as robust.
2. *Classes*: not suitable s < 0.5, marginal 0.5–0.7, suitable 0.7–0.9, highly suitable ≥ 0.9 (`CLASS_EDGES`). Land
   (Mha) and share of today's crop area per class, ERA5 and each run; transitions no hosing → hosing.
3. *Classes of 12.b* (function of 12.b, recomputed here): for each class (suitable in both, robust / uncertain loss and
   gain), the index without and with hosing, Δs, and the share of the cells whose change of the index is robust.

**Reading of 3.** If the supervisor's reading holds, the robust losses have a large Δs (from well above 0.5 to well
below), the uncertain losses an index close to 0.5 in both runs and a small Δs (degraded towards the margin). The
cells *suitable in both* with a robust negative Δs are the land that stays suitable but degrades.

Limits: one model, one member per run; the index is a probability of the classifier, its classes are thresholds of
convenience; the noise per cell is estimated from four pairs of halves.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
import warnings; warnings.filterwarnings('ignore'); os.environ["PYTHONWARNINGS"] = "ignore"   # also in the joblib workers

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 1.3.a, 4.g and 6.b
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = "no_rad"
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
PARTS = ["full", "H1", "H2"]
LEVELS = ["low", "high"]
MAP_LEVEL = "high"
CLASS_EDGES = [0.5, 0.7, 0.9]     # not suitable < 0.5 <= marginal < 0.7 <= suitable < 0.9 <= highly suitable
CLASS_NAMES = ["not suitable", "marginal", "suitable", "highly suitable"]
MIN_CHANGE = 0.1                  # smallest change of the index counted as robust (the noise is ~0 near 0 and 1)
KERNEL_EPS = 0.01
N_JOBS = min(5, os.cpu_count() or 1)
assert CHOSEN_RAD == "no_rad", "no_rad only (no rsds in the runs)"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
BIO12A = cu.out_dir(CFG, "test_internal_variability", ROUTE, SET) / "bio"    # 12.a: delta BIO of the runs and halves
OUT = cu.out_dir(CFG, "test_probability_change", ROUTE, SET)                # the only folder written

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
DT = cu.tmin_warming(CFG)
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
era5_file = lambda crop: ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
half_file = lambda run, part, crop: BIO12A / f"BIOvariables_delta_{run}_{part}_{crop}.nc"
with xr.open_dataset(era5_file(CROPS[0])) as d0:
    COLS = [v for v in d0.data_vars if v.startswith("BIO") and not v.endswith("e") and v not in ("BIO03", "BIO08")]
miss = [str(half_file(r, p, c)) for r in RUNS for p in PARTS for c in CROPS if not half_file(r, p, c).exists()]
assert not miss, f"delta BIO of 12.a missing (run 12.a first): {miss[:3]}"
print(f"set {SET} | runs {RUNS} | classes {CLASS_EDGES} | robust change >= {MIN_CHANGE} | written only in {OUT}")

## 1. Suitability index of ERA5, of the whole runs and of their halves (function of 12.a, with the probability)

In [ ]:
def suitability(crop):
    # {"ERA5" or (run, part): {"suit": {level: bool}, "novel": bool, "p": probability at equal priors, 0 where
    #  novel, "floor": {level: bool, ECOCROP TMIN and soil >= 10}}}: the function of 12.a with p and the floors (as 12.e)
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, n1, n0 = bundle["bio_cols"], bundle["n_class1"], bundle["n_class0"]
    assert set(cols) <= set(COLS), (cols, COLS)
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(era5_file(crop)) as ds:
        REF = cu.bio_array(ds, cols, CFG)[ii, jj]
    REF = REF[np.isfinite(REF).all(axis=1)]
    kfun = cu.kernel_support(bundle["model"])
    Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float),
                            tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
    ssr = {lv: np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0) for lv in LEVELS}
    which = cu.ECOCROP_TMIN[crop][1]
    files = {"ERA5": era5_file(crop), **{(r, p): half_file(r, p, crop) for r in RUNS for p in PARTS}}
    out = {}
    for key, path in files.items():
        with xr.open_dataset(path) as ds:
            A = cu.bio_array(ds, cols, CFG)
            tg = np.asarray(cu.growing_temperature(ds, which), float)
        X = A.reshape(-1, len(cols))
        ok = np.isfinite(X).all(axis=1) & LAND.ravel()
        p, nov = np.full(X.shape[0], np.nan), np.zeros(X.shape[0], bool)
        if ok.any():                                              # (a part without any season: all NaN)
            Xs = bundle["scaler"].transform(X[ok])
            p[ok] = bundle["model"].predict_proba(Xs)[:, 1]
            nov[ok] = (cu.mess(X[ok], REF)[0] < 0) | (cu.nt2(X[ok], REF) > 1) | (kfun(Xs) < KERNEL_EPS)
        p, nov = p.reshape(LAND.shape), nov.reshape(LAND.shape)
        g = cu.crossfit_proba(Fd, A[ii, jj])                      # training cells: the fold model that did not see them
        fin = np.isfinite(g)
        p[ii[fin], jj[fin]] = g[fin]
        p = cu.balanced_proba(p, n1, n0)
        nov &= np.isfinite(p)
        proba = np.where(nov, 0.0, p)
        clim = cu.climate_suitable(proba, tg, crop, DT) & np.isfinite(proba)
        tmin_ok = cu.climate_suitable(np.ones(LAND.shape), tg, crop, DT)           # the ECOCROP TMIN alone
        out[key] = {"suit": {lv: LAND & cu.suitable(clim, ssr[lv]) for lv in LEVELS}, "novel": nov,
                    "p": np.where(LAND & np.isfinite(proba), proba, 0.0),
                    "floor": {lv: LAND & tmin_ok & cu.suitable(np.ones(LAND.shape), ssr[lv]) for lv in LEVELS}}
    return out


def classes(R, h, o, lv):
    full = R[(h, "full")]["suit"][lv].astype(int) - R[(o, "full")]["suit"][lv].astype(int)
    ch = {p: R[(h, p)]["suit"][lv].astype(int) - R[(o, p)]["suit"][lv].astype(int) for p in ("H1", "H2")}
    both = R[(h, "full")]["suit"][lv] & R[(o, "full")]["suit"][lv]
    c = np.where(both, 1, 0)
    agree = lambda v: (full == v) & (ch["H1"] == v) & (ch["H2"] == v)
    c = np.where(full == -1, np.where(agree(-1), 2, 4), c)
    c = np.where(full == 1, np.where(agree(1), 3, 5), c)
    nov = R[(h, "full")]["novel"] | R[(o, "full")]["novel"]
    return np.where(LAND, np.where(nov, 6, c), -1)


mha = lambda m: float(np.nansum(LAND_MHA * m))
index = lambda c, lv: np.where(c["floor"][lv], c["p"], 0.0)                 # suitability index s
cls = lambda s: np.digitize(s, CLASS_EDGES)                                  # 0 not suitable .. 3 highly suitable
B12 = ["neither", "both", "robust loss", "robust gain", "uncertain loss", "uncertain gain", "novel in either"]

rows_cl, rows_tr, rows_b, rows_d, MAPS, check = [], [], [], [], {}, []
for crop in CROPS:
    t0 = time.time()
    R = suitability(crop)
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    for lv in LEVELS:
        S_ = {k: index(c, lv) for k, c in R.items()}
        for k, c in R.items():                                            # check: s >= 0.5 is the binary of 6.b
            check.append(int(((S_[k] >= cu.CLIMATE_P) != c["suit"][lv])[LAND].sum()))
        # (2) areas per class
        for k in ["ERA5", *[(r, "full") for r in RUNS]]:
            c = cls(S_[k])
            for j, n in enumerate(CLASS_NAMES):
                rows_cl.append({"crop": crop, "level": lv, "climate": k if k == "ERA5" else k[0], "class": n,
                                "land (Mha)": mha(LAND & (c == j)), "today's crop area (%)": share(LAND & (c == j))})
        for h, o in PAIRS:
            sh, so = S_[(h, "full")], S_[(o, "full")]
            nov = R[(h, "full")]["novel"] | R[(o, "full")]["novel"]
            ok = LAND & ~nov
            ch, co = cls(sh), cls(so)
            for a in range(4):                                            # transitions no hosing -> hosing
                for b in range(4):
                    rows_tr.append({"crop": crop, "level": lv, "pair": f"{h} - {o}", "from (no hosing)": CLASS_NAMES[a],
                                    "to (hosing)": CLASS_NAMES[b], "land (Mha)": mha(ok & (co == a) & (ch == b)),
                                    "today's crop area (%)": share(ok & (co == a) & (ch == b))})
            # (1) change of the index and its robustness per cell
            ds_ = {p: S_[(h, p)] - S_[(o, p)] for p in PARTS}
            d = np.stack([S_[(r, "H1")] - S_[(r, "H2")] for r in RUNS])
            se = np.sqrt(np.mean(d ** 2, axis=0) / 2)
            same = (np.sign(ds_["full"]) == np.sign(ds_["H1"])) & (np.sign(ds_["full"]) == np.sign(ds_["H2"])) & (ds_["full"] != 0)
            rob = ok & same & (np.abs(ds_["full"]) > 2 * se) & (np.abs(ds_["full"]) >= MIN_CHANGE)
            if lv == MAP_LEVEL:
                MAPS[(crop, h)] = (np.where(ok & ((sh > 0) | (so > 0)), ds_["full"], np.nan), rob)
            # (3) the classes of 12.b in terms of the index
            c12 = classes(R, h, o, lv)
            for j, n in enumerate(B12[:6]):
                m = (c12 == j) & ok
                if j == 0:
                    continue
                rows_b.append({"crop": crop, "level": lv, "pair": f"{h} - {o}", "class of 12.b": n, "cells": int(m.sum()),
                               "land (Mha)": mha(m), "today's crop area (%)": share(m),
                               "index, no hosing (median)": float(np.median(so[m])) if m.any() else np.nan,
                               "index, hosing (median)": float(np.median(sh[m])) if m.any() else np.nan,
                               "change of the index (median)": float(np.median(ds_["full"][m])) if m.any() else np.nan,
                               "|change| / SE (median)": float(np.median(np.abs(ds_["full"][m]) / np.maximum(se[m], 1e-6))) if m.any() else np.nan,
                               "robust change of the index (% of the cells)": 100 * float(rob[m].mean()) if m.any() else np.nan,
                               "robust decrease (% of the cells)": 100 * float((rob & (ds_["full"] < 0))[m].mean()) if m.any() else np.nan,
                               "index within 0.4-0.6 in both runs (% of the cells)": 100 * float(((np.abs(so - 0.5) < 0.1) & (np.abs(sh - 0.5) < 0.1))[m].mean()) if m.any() else np.nan})
            for name, m in (("all land", ok & ((sh > 0) | (so > 0))), ("today's cropland", ok & (crop_ha > 0))):
                rows_d.append({"crop": crop, "level": lv, "pair": f"{h} - {o}", "domain": name,
                               "robust decrease (Mha)": mha(m & rob & (ds_["full"] < 0)), "robust increase (Mha)": mha(m & rob & (ds_["full"] > 0)),
                               "robust decrease, suitable in both (Mha)": mha(m & rob & (ds_["full"] < 0) & (so >= 0.5) & (sh >= 0.5)),
                               "robust decrease, today's crop area (%)": share(m & rob & (ds_["full"] < 0)),
                               "robust increase, today's crop area (%)": share(m & rob & (ds_["full"] > 0)),
                               "mean change of the index, today's cropland (weighted by the crop area)":
                                   float((crop_ha * ds_["full"] * ok).sum() / max(crop_ha[ok].sum(), 1e-9))})
    del R
    print(f"{crop}: {(time.time() - t0) / 60:.1f} min", flush=True)
print("check, index >= 0.5 against the binary of 6.b: largest number of differing cells", max(check))

## 2. Tables: classes, transitions, the classes of 12.b, robust changes of the index

In [ ]:
CL = pd.DataFrame(rows_cl).set_index(["crop", "level", "climate", "class"])
TR = pd.DataFrame(rows_tr)
B = pd.DataFrame(rows_b).set_index(["crop", "level", "pair", "class of 12.b"])
DD = pd.DataFrame(rows_d).set_index(["crop", "level", "pair", "domain"])
CL.to_csv(OUT / "areas_by_class.csv"); TR.to_csv(OUT / "class_transitions.csv", index=False)
B.to_csv(OUT / "classes_12b_vs_index.csv"); DD.to_csv(OUT / "robust_index_change.csv")
last = f"{PAIRS[-1][0]} - {PAIRS[-1][1]}"
with pd.option_context("display.max_rows", 500, "display.width", 250, "display.max_columns", 30):
    print(f"(2) Areas per class, {MAP_LEVEL} input (Mha)")
    display(CL.xs(MAP_LEVEL, level="level")["land (Mha)"].unstack("class")[CLASS_NAMES].round(0))
    print(f"(2) Today's crop area per class, {MAP_LEVEL} input (%)")
    display(CL.xs(MAP_LEVEL, level="level")["today's crop area (%)"].unstack("class")[CLASS_NAMES].round(1))
    t = TR[(TR.level == MAP_LEVEL) & (TR.pair == last)]
    print(f"(2) Transitions {last}, {MAP_LEVEL} input, today's crop area (%): rows no hosing, columns hosing")
    for crop in CROPS:
        print(crop); display(t[t.crop == crop].pivot(index="from (no hosing)", columns="to (hosing)", values="today's crop area (%)").loc[CLASS_NAMES, CLASS_NAMES].round(1))
    print(f"(3) The classes of 12.b in terms of the index, {last}, {MAP_LEVEL} input")
    display(B.xs((MAP_LEVEL, last), level=("level", "pair")).round(2))
    print(f"(1) Robust changes of the index, {MAP_LEVEL} input")
    display(DD.xs(MAP_LEVEL, level="level").round(2))

## 3. Maps: change of the index in 2050, robust cells (|change| ≥ MIN_CHANGE) strong colours, the others pale

In [ ]:
h = PAIRS[-1][0]
fig, axes = plt.subplots((len(CROPS) + 1) // 2, 2, figsize=(26, 5.6 * ((len(CROPS) + 1) // 2)), squeeze=False)
cmap = plt.get_cmap("BrBG")
for ax, crop in zip(axes.ravel(), CROPS):
    dsv, rob = MAPS[(crop, h)]
    ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 0, np.nan), cmap="Greys", vmin=0, vmax=8, shading="nearest")
    ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(rob, np.nan, dsv), cmap=cmap, vmin=-0.5, vmax=0.5, alpha=0.35, shading="nearest")
    im = ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(rob, dsv, np.nan), cmap=cmap, vmin=-0.5, vmax=0.5, shading="nearest")
    r = DD.loc[(crop, MAP_LEVEL, f"{h} - {PAIRS[-1][1]}", "all land")]
    ax.set_ylim(-60, 85)
    ax.set_title(f"{crop}, {MAP_LEVEL} input, 2050 hosing - no hosing: change of the suitability index; robust decrease "
                 f"{r['robust decrease (Mha)']:.0f} Mha (of which suitable in both {r['robust decrease, suitable in both (Mha)']:.0f}), "
                 f"increase {r['robust increase (Mha)']:.0f} Mha", fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8, label="change of the index")
for ax in axes.ravel()[len(CROPS):]:
    ax.axis("off")
fig.tight_layout(); fig.savefig(OUT / f"index_change_2050_{MAP_LEVEL}.png", dpi=90, bbox_inches="tight"); plt.show()
print("written:", sorted(p.name for p in OUT.iterdir()))